"""
Hull Tactical Market Prediction — XGBoost/LightGBM Ensemble

Design:
- Advanced gradient boosting ensemble for alpha prediction:
  * XGBoost, LightGBM, CatBoost (diverse tree-based models)
  * ElasticNet (linear baseline)
  * PCA + Ridge (dimensionality reduction)
- Rolling feature engineering: moving averages, volatility, momentum, z-scores
  over multiple time windows (5, 10, 21, 63 days)
- OOF stacking: Ridge meta-learner on out-of-fold predictions (no leakage)
- Position mapping: choose slope k and EMA smoothing via walk-forward CV
  under a 1.2× vol cap vs market vol. Final k, smoothing are medians over folds.
- Inference: blended alpha -> pos_raw = 1 + k*alpha -> EMA smoothing -> clip [0,2]
- Online-safe: keeps only last position for smoothing state; no label peeking.

No internet used. Uses only /kaggle/input/hull-tactical-market-prediction/.
"""


In [ ]:
from __future__ import annotations
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from typing import List, Optional

# Gradient Boosting Libraries
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

# Sklearn for preprocessing and meta-learner
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.decomposition import PCA
from sklearn.base import BaseEstimator, RegressorMixin, clone

# Kaggle evaluation API
import kaggle_evaluation.default_inference_server


In [ ]:
# ----------------------------
# Config
# ----------------------------
DATA_DIR = "/kaggle/input/hull-tactical-market-prediction/"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")

RANDOM_STATE = 42
VAL_SIZE = 180          # walk-forward validation window (~6 months)
N_FOLDS = 6             # number of walk-forward folds
GAP = 5                 # small gap to reduce leakage
VOL_CAP_RATIO = 1.2     # portfolio vol cap vs market vol
MIN_COVERAGE = 0.20     # keep features with >=20% non-missing ratio

# EMA smoothing grid (for turnover control)
EMA_GRID = [0.00, 0.10, 0.20, 0.30, 0.50, 0.70]  # alpha for EMA on positions; 0=no smoothing

# Mapping slope grid k
K_GRID = np.concatenate([
    np.linspace(0.0, 5.0, 26),
    np.linspace(6.0, 20.0, 15),
    np.linspace(25.0, 50.0, 11),
    np.linspace(60.0, 150.0, 10),
    np.linspace(200.0, 500.0, 16),
])
K_GRID = np.unique(np.round(K_GRID, 6))

# Rolling window sizes for feature engineering
ROLLING_WINDOWS = [5, 10, 21, 63]  # short, medium, long-term


In [ ]:
# ----------------------------
# Helpers
# ----------------------------

def build_lagged_labels(df: pd.DataFrame) -> pd.DataFrame:
    """Add lagged target columns to match test schema."""
    df = df.sort_values("date_id").copy()
    for col in ["forward_returns", "risk_free_rate", "market_forward_excess_returns"]:
        df[f"lagged_{col}"] = df[col].shift(1)
    return df


def create_rolling_features(df: pd.DataFrame, feature_cols: List[str], windows: List[int]) -> pd.DataFrame:
    """Create rolling statistics features."""
    df = df.copy()
    for col in feature_cols:
        for window in windows:
            if len(df) < window:
                continue
                
            # Rolling mean
            df[f"{col}_roll_mean_{window}"] = df[col].rolling(window=window, min_periods=1).mean()
            df[f"{col}_roll_std_{window}"] = df[col].rolling(window=window, min_periods=1).std()
            df[f"{col}_momentum_{window}"] = df[col] - df[f"{col}_roll_mean_{window}"]
            roll_std = df[f"{col}_roll_std_{window}"]
            df[f"{col}_zscore_{window}"] = df[f"{col}_momentum_{window}"] / (roll_std + 1e-8)
    return df


def select_features(df: pd.DataFrame, min_non_missing_ratio: float = MIN_COVERAGE) -> List[str]:
    """Select features with sufficient coverage and non-zero variance."""
    exclude = {"date_id", "forward_returns", "risk_free_rate", 
               "market_forward_excess_returns", "is_scored"}
    cols = [c for c in df.columns if c not in exclude]
    cov = 1.0 - df[cols].isna().mean()
    keep = list(cov[cov >= min_non_missing_ratio].index)
    nunique = df[keep].nunique(dropna=False)
    return [c for c in keep if nunique[c] > 1]


def walk_forward_splits(date_ids: np.ndarray, n_folds: int = N_FOLDS, 
                        val_size: int = VAL_SIZE, gap: int = GAP):
    """Generate walk-forward time series splits."""
    N = len(date_ids)
    splits = []
    for i in range(n_folds, 0, -1):
        val_end = N - (i - 1) * val_size
        val_start = val_end - val_size
        if val_start <= 0:
            continue
        train_end = max(0, val_start - gap)
        tr = np.arange(0, train_end)
        va = np.arange(val_start, val_end)
        if len(tr) > 200 and len(va) == val_size:
            splits.append((tr, va))
    return splits[-n_folds:] if splits else [(np.arange(0, max(0, N - val_size - gap)), 
                                               np.arange(max(0, N - val_size), N))]


def apply_vol_cap(pos_minus_one: np.ndarray, fwd_returns: np.ndarray, 
                  cap_ratio: float = VOL_CAP_RATIO) -> np.ndarray:
    """Scale positions to respect volatility cap."""
    mkt_vol = np.nanstd(fwd_returns)
    if mkt_vol <= 0:
        return pos_minus_one
    port_vol = np.nanstd(pos_minus_one * fwd_returns)
    if port_vol == 0:
        return pos_minus_one
    max_port = cap_ratio * mkt_vol
    if port_vol <= max_port:
        return pos_minus_one
    return pos_minus_one * (max_port / port_vol)


def ema_smooth(prev_pos: float, raw_pos: float, alpha: float) -> float:
    """Exponential moving average smoothing."""
    if alpha <= 0:
        return raw_pos
    return (1 - alpha) * prev_pos + alpha * raw_pos


In [ ]:
# ----------------------------
# Model Builders
# ----------------------------

def make_xgboost_model():
    """XGBoost with moderate regularization."""
    return Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("xgb", xgb.XGBRegressor(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=5,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            tree_method="hist",
        ))
    ])


def make_lightgbm_model():
    """LightGBM with moderate regularization."""
    return Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("lgb", lgb.LGBMRegressor(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_samples=20,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        ))
    ])


def make_catboost_model():
    """CatBoost with moderate regularization."""
    return Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("cat", cb.CatBoostRegressor(
            iterations=300,
            depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bylevel=0.8,
            min_data_in_leaf=20,
            l2_leaf_reg=3.0,
            random_seed=RANDOM_STATE,
            verbose=False,
            thread_count=-1,
        ))
    ])


def make_elasticnet_model():
    """ElasticNet baseline."""
    return Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
        ("en", ElasticNet(alpha=5e-4, l1_ratio=0.2, max_iter=30000, random_state=RANDOM_STATE))
    ])


def make_pca_ridge_model(n_components: int = 20):
    """PCA + Ridge for dimensionality reduction."""
    return Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
        ("pca", PCA(n_components=n_components, random_state=RANDOM_STATE)),
        ("ridge", Ridge(alpha=1.0, random_state=RANDOM_STATE))
    ])




In [ ]:
# ----------------------------
# OOF Stacking Blender
# ----------------------------

class OOFStackBlender(BaseEstimator, RegressorMixin):
    """Out-of-fold stacking ensemble."""
    def __init__(self, meta_alpha=1.0):
        self.base_specs = [
            ("xgb", make_xgboost_model()),
            ("lgb", make_lightgbm_model()),
            ("cat", make_catboost_model()),
            ("elastic", make_elasticnet_model()),
            ("pca_ridge", make_pca_ridge_model(n_components=20))
        ]
        self.meta = Ridge(alpha=meta_alpha, random_state=RANDOM_STATE)
        self.fitted_bases_ = {}

    def fit(self, X: pd.DataFrame, y: np.ndarray, date_ids: np.ndarray, splits):
        """Train base models and meta-learner using walk-forward CV."""
        n_samples = len(X)
        n_models = len(self.base_specs)
        oof_preds = np.zeros((n_samples, n_models), dtype=float)
        oof_mask = np.zeros(n_samples, dtype=bool)

        print(f"Training {n_models} base models on {len(splits)} folds...", flush=True)
        
        for fold, (tr, va) in enumerate(splits):
            Xtr, ytr = X.iloc[tr], y[tr]
            Xva = X.iloc[va]
            
            for j, (name, model) in enumerate(self.base_specs):
                mdl = clone(model)
                mdl.fit(Xtr, ytr)
                oof_preds[va, j] = mdl.predict(Xva)
            
            oof_mask[va] = True
            print(f"  Completed fold {fold + 1}/{len(splits)}", flush=True)

        # Train meta-learner
        valid_mask = oof_mask & ~np.isnan(y)
        self.meta.fit(oof_preds[valid_mask], y[valid_mask])
        print("Meta-learner trained.", flush=True)

        # Refit base models on full data
        print("Refitting base models on full data...", flush=True)
        valid_y = ~np.isnan(y)
        for name, model in self.base_specs:
            mdl = clone(model)
            mdl.fit(X[valid_y], y[valid_y])
            self.fitted_bases_[name] = mdl
        print("Base models refitted.", flush=True)

        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        """Predict using stacked ensemble."""
        base_preds = [self.fitted_bases_[name].predict(X) for name, _ in self.base_specs]
        base_mat = np.column_stack(base_preds)
        return self.meta.predict(base_mat)


In [ ]:
# ----------------------------
# Main Forecast Model
# ----------------------------

class ForecastEnsemble:
    """
    Complete forecasting pipeline with:
    - Feature engineering (rolling features)
    - Gradient boosting ensemble (XGBoost, LightGBM, CatBoost)
    - OOF stacking
    - Position mapping with volatility control
    - EMA smoothing
    """
    def __init__(self, use_rolling_features: bool = True):
        self.use_rolling_features = use_rolling_features
        self.features_: Optional[List[str]] = None
        self.original_features_: Optional[List[str]] = None
        self.blender_: Optional[OOFStackBlender] = None
        self.k_: float = 1.0
        self.ema_alpha_: float = 0.0
        self.last_position_: float = 1.0
        self.fitted_: bool = False

    def fit(self, path: str = TRAIN_CSV):
        """Train the complete model pipeline."""
        print("Loading and preprocessing data...", flush=True)
        df = pd.read_csv(path)
        df = build_lagged_labels(df)
        
        # Initial feature selection (before rolling features)
        self.original_features_ = select_features(df, MIN_COVERAGE)
        
        # Create rolling features if enabled
        if self.use_rolling_features:
            print(f"Creating rolling features from {len(self.original_features_)} base features...", flush=True)
            # Select top features for rolling (to avoid explosion)
            y_temp = df["market_forward_excess_returns"].values
            corrs = []
            for feat in self.original_features_:
                corr = np.corrcoef(df[feat].fillna(0).values, y_temp)[0, 1]
                if np.isfinite(corr):
                    corrs.append((feat, abs(corr)))
            
            # Select top 30 features for rolling feature engineering
            top_features = sorted(corrs, key=lambda x: x[1], reverse=True)[:30]
            top_feature_names = [f[0] for f in top_features]
            
            df = create_rolling_features(df, top_feature_names, ROLLING_WINDOWS)
            print(f"  Added rolling features. Total columns: {len(df.columns)}", flush=True)
        
        # Final feature selection (after rolling features)
        self.features_ = select_features(df, MIN_COVERAGE)
        
        X = df[self.features_].copy()
        y = df["market_forward_excess_returns"].astype(float).values
        fwd = df["forward_returns"].astype(float).values
        date_ids = df["date_id"].values

        print(f"Final feature count: {len(self.features_)}", flush=True)

        # Generate walk-forward splits
        splits = walk_forward_splits(date_ids, N_FOLDS, VAL_SIZE, GAP)
        print(f"Using {len(splits)} walk-forward folds", flush=True)

        # Train stacking ensemble
        self.blender_ = OOFStackBlender(meta_alpha=1.0)
        self.blender_.fit(X, y, date_ids, splits)
        print("Stacking ensemble training complete.", flush=True)

        # Choose mapping parameters (k, ema_alpha) via CV
        print("Selecting position mapping parameters...")
        best_pairs = []
        
        for fold, (tr, va) in enumerate(splits):
            # Get OOF predictions for this fold
            yhat_va = self.blender_.predict(X.iloc[va])
            fwd_va = fwd[va]
            y_va = y[va]
            
            # Check prediction scale
            pred_mean = np.nanmean(np.abs(yhat_va))
            pred_std = np.nanstd(yhat_va)
            if fold == 0:
                print(f"  Prediction scale: mean|pred|={pred_mean:.6f}, std={pred_std:.6f}")

            best_score, best_k, best_ema = -1e18, 1.0, 0.0
            
            # Sample grid for speed (every 3rd k value if grid is large)
            k_sample = K_GRID[::3] if len(K_GRID) > 50 else K_GRID
            
            for ema_a in EMA_GRID:
                for k in k_sample:
                    pm1 = k * yhat_va
                    pm1 = apply_vol_cap(pm1, fwd_va, cap_ratio=VOL_CAP_RATIO)
                    pos_raw = 1.0 + pm1
                    
                    # Apply EMA smoothing online
                    smoothed = np.empty_like(pos_raw)
                    prev = 1.0
                    for t, pr in enumerate(pos_raw):
                        smoothed[t] = ema_smooth(prev, pr, ema_a)
                        prev = smoothed[t]
                    
                    pos = np.clip(smoothed, 0.0, 2.0)
                    
                    # Objective: mean((pos-1)*y) - aligned with alpha target
                    score = float(np.nanmean((pos - 1.0) * y_va))
                    
                    if score > best_score:
                        best_score, best_k, best_ema = score, float(k), float(ema_a)
            
            best_pairs.append((best_k, best_ema))
            print(f"  Fold {fold+1}: k={best_k:.3f}, ema={best_ema:.2f}, score={best_score:.6f}")

        # Robust aggregate (median)
        self.k_ = float(np.median([p[0] for p in best_pairs]))
        self.ema_alpha_ = float(np.median([p[1] for p in best_pairs]))
        
        self.fitted_ = True
        print(f"\n[Fit Complete]", flush=True)
        print(f"  Features: {len(self.features_)}", flush=True)
        print(f"  Mapping k: {self.k_:.3f}", flush=True)
        print(f"  EMA alpha: {self.ema_alpha_:.2f}", flush=True)
        print(f"  Folds: {len(splits)}", flush=True)

    def predict_positions(self, batch_df) -> np.ndarray:
        """Predict positions for new data."""
        # Convert polars to pandas if needed
        try:
            import polars as pl
            if isinstance(batch_df, pl.DataFrame):
                batch_df = batch_df.to_pandas()
        except:
            pass

        # Align columns
        X = pd.DataFrame(index=batch_df.index)
        for c in self.features_:
            X[c] = batch_df[c] if c in batch_df.columns else np.nan

        # Predict alpha and map to positions
        alpha = self.blender_.predict(X)
        pm1 = self.k_ * alpha
        pos_raw = 1.0 + pm1
        
        # Apply EMA smoothing
        out = np.empty_like(pos_raw)
        prev = self.last_position_
        for i, pr in enumerate(pos_raw):
            out[i] = ema_smooth(prev, pr, self.ema_alpha_)
            prev = out[i]
        
        self.last_position_ = float(prev)
        return np.clip(out, 0.0, 2.0).astype(float)


In [ ]:
# ----------------------------
# Kaggle Evaluation API
# ----------------------------

_MODEL: Optional[ForecastEnsemble] = None

def _ensure_model():
    """Lazy initialization of model."""
    global _MODEL
    if _MODEL is None:
        _MODEL = ForecastEnsemble(use_rolling_features=True)
        _MODEL.fit(TRAIN_CSV)

def predict(data_batch):
    """
    Kaggle evaluation API predict endpoint.
    
    Parameters
    ----------
    data_batch : pandas.DataFrame or polars.DataFrame
        Features for one or more timesteps.
    
    Returns
    -------
    float | np.ndarray
        Position(s) in [0, 2]. Scalar for single row, array for multiple.
    """
    _ensure_model()

    # Convert polars to pandas if needed
    try:
        import polars as pl
        if isinstance(data_batch, pl.DataFrame):
            data_batch = data_batch.to_pandas()
    except:
        pass

    pos_vec = _MODEL.predict_positions(data_batch)
    
    # Return scalar for single-row batch
    if getattr(data_batch, "shape", None) and data_batch.shape[0] == 1:
        return float(pos_vec[0])
    
    return pos_vec




In [ ]:
# ----------------------------
# Start Inference Server
# ----------------------------

inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    # Production: serve the API
    inference_server.serve()
else:
    # Local testing: generate submission.parquet first, then run gateway
    TEST_CSV = os.path.join(DATA_DIR, "test.csv")
    LOCAL_TEST_CSV = "hull-tactical-market-prediction/test.csv"
    if os.path.exists(LOCAL_TEST_CSV):
        TEST_CSV = LOCAL_TEST_CSV
    
    if os.path.exists(TEST_CSV):
        print("Generating submission.parquet...")
        # Train model once
        _ensure_model()
        # Generate predictions
        test_df = pd.read_csv(TEST_CSV)
        positions = _MODEL.predict_positions(test_df)
        submission_df = pd.DataFrame({'position': positions})
        submission_df.to_parquet('submission.parquet', index=False)
        print(f"Saved submission.parquet with {len(submission_df)} predictions")
        print(f"Position stats: min={positions.min():.4f}, max={positions.max():.4f}, mean={positions.mean():.4f}, std={positions.std():.4f}")
    else:
        print("Warning: test.csv not found, skipping submission.parquet generation")
    
    # Run gateway for validation (model already trained)
    print("\nRunning local gateway against public data...")
    inference_server.run_local_gateway((DATA_DIR,))

inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    # Production: serve the API
    inference_server.serve()
else:
    # Local testing: generate submission.parquet first, then run gateway
    TEST_CSV = os.path.join(DATA_DIR, "test.csv")
    if os.path.exists(TEST_CSV):
        print("Generating submission.parquet...")
        # Train model once
        _ensure_model()
        # Generate predictions
        test_df = pd.read_csv(TEST_CSV)
        positions = _MODEL.predict_positions(test_df)
        submission_df = pd.DataFrame({'position': positions})
        submission_df.to_parquet('submission.parquet', index=False)
        print(f"Saved submission.parquet with {len(submission_df)} predictions")
        print(f"Position stats: min={positions.min():.4f}, max={positions.max():.4f}, mean={positions.mean():.4f}")
    else:
        print("Warning: test.csv not found, skipping submission.parquet generation")
    
    # Run gateway for validation (model already trained)
    print("\nRunning local gateway against public data...")
    inference_server.run_local_gateway((DATA_DIR,))
